# Linguagens de Programação para Engenharia de Dados
## Encontro 2 — Notebook Prático: Manipulação de dados com pandas

**Instituição:** Universidade de Fortaleza (UNIFOR)
**Curso:** Pós-Graduação — Especialização em Engenharia de Dados
**Professor:** Cassio Pinheiro — [LinkedIn](https://www.linkedin.com/in/cassio-pinheiro-9baa7939/)
**Data:** 26/06/2026 · 19:00 às 22:30

---

### Como usar este notebook

1. Execute as células **na ordem**, de cima para baixo (`Shift + Enter`).
2. Cada seção corresponde a uma seção do documento `.md` do encontro.
3. O Encontro 2 roda com **pandas** (obrigatório) e **pyarrow** (para Parquet). A seção 6 tem um trecho **opcional** com `polars` — se não estiver instalado, o notebook segue normalmente com um aviso.
4. **Mexa no código.** Troque valores, quebre de propósito, conserte. É assim que se aprende.

> **Projeto fio condutor:** o **mesmo** `vendas.csv` sujo do Encontro 1 (datas em 3 formatos, valor ausente no id 3, negativo no id 4, duplicata no id 5, data vazia no id 7, espaços no id 2). Hoje refazemos o pipeline do E1 **em pandas**, geramos **Parquet** e provamos que o faturamento bate com o do E1.

## Setup — pandas, pyarrow e (opcional) polars

Instale, se necessário, com `pip install pandas pyarrow polars`. O `polars` é opcional: o notebook detecta sua ausência e segue.

In [1]:
import io
import time
import pandas as pd

print("pandas:", pd.__version__)

# Polars é OPCIONAL — tentamos importar; se faltar, seguimos sem ele.
try:
    import polars as pl
    TEM_POLARS = True
    print("polars:", pl.__version__)
except ImportError:
    TEM_POLARS = False
    print("polars não instalado — a comparação da seção 6 será apenas demonstrada.")

print("\nSetup OK — pandas entra em cena a partir deste encontro.")

pandas: 2.3.3
polars: 1.42.0

Setup OK — pandas entra em cena a partir deste encontro.


In [2]:
# CSV sintético SUJO de propósito — É O MESMO do Encontro 1.
# Representa o que chega de um sistema legado, com todos os problemas reais.
csv_bruto = """id,data,cliente,categoria,valor
1,2026-06-01,Ana Souza,eletronicos,1200.00
2,01/06/2026,  Bruno Lima ,alimentacao,85.50
3,2026-06-02,Carla Dias,eletronicos,
4,2026-06-02,Diego Reis,transporte,-30.00
5,2026/06/03,Ana Souza,alimentacao,42.90
5,2026/06/03,Ana Souza,alimentacao,42.90
6,2026-06-03,Elena Cruz,vestuario,260.00
7,,Felipe Aragao,servicos,99.90
"""

print("Problemas plantados: data em 3 formatos, valor vazio (id 3),")
print("valor negativo (id 4), linha duplicada (id 5), data vazia (id 7),")
print("e espaços sobrando no cliente (id 2).")

Problemas plantados: data em 3 formatos, valor vazio (id 3),
valor negativo (id 4), linha duplicada (id 5), data vazia (id 7),
e espaços sobrando no cliente (id 2).


---

# 1 — Por que pandas

> 📖 **No `.md`:** seção *1 — Por que pandas*.

No E1 somávamos colunas com um laço Python interpretado. O pandas faz a mesma soma de forma **vetorizada** — uma operação sobre a coluna inteira, executada em C sobre arrays NumPy. Vamos **medir** a diferença no mesmo dado.

In [3]:
# Geramos uma coluna grande (1 milhão de números) para a diferença ficar visível.
import random
random.seed(42)               # determinístico: mesmo resultado toda vez
valores = [random.random() * 100 for _ in range(1_000_000)]

# (a) LAÇO PYTHON PURO — estilo Encontro 1
inicio = time.perf_counter()
soma_loop = 0.0
for v in valores:
    soma_loop += v
t_loop = time.perf_counter() - inicio

# (b) VETORIZADO com pandas — uma Series, uma operação
serie = pd.Series(valores)
inicio = time.perf_counter()
soma_vet = serie.sum()
t_vet = time.perf_counter() - inicio

print(f"Soma (laço Python) : {soma_loop:,.2f}  em {t_loop*1000:8.2f} ms")
print(f"Soma (pandas .sum()): {soma_vet:,.2f}  em {t_vet*1000:8.2f} ms")
print(f"\nMesmo resultado: {abs(soma_loop - soma_vet) < 1e-3}")
print(f"pandas foi ~{t_loop / t_vet:.0f}x mais rápido nesta máquina.")
print("\nA lição: em pandas, pense em COLUNAS INTEIRAS, não em laços sobre linhas.")

Soma (laço Python) : 50,001,454.39  em    22.36 ms
Soma (pandas .sum()): 50,001,454.39  em     0.70 ms

Mesmo resultado: True
pandas foi ~32x mais rápido nesta máquina.

A lição: em pandas, pense em COLUNAS INTEIRAS, não em laços sobre linhas.


---

# 2 — DataFrame e Series

> 📖 **No `.md`:** seção *2 — DataFrame e Series*.

Lemos o `vendas.csv` sujo para um **DataFrame** e fazemos o ritual de chegada: `head`, `info`, `describe`. Repare que, lido como texto, os **dtypes ainda estão errados** — é exatamente o que vamos corrigir na seção 4.

In [4]:
# Lemos o CSV bruto a partir de uma string, via io.StringIO (simula abrir um arquivo).
# dtype=str: forçamos TUDO como texto, para ver o dado exatamente como chegou.
df_bruto = pd.read_csv(io.StringIO(csv_bruto), dtype=str)

print("df_bruto é um", type(df_bruto).__name__, "com shape", df_bruto.shape)
print("\nUma coluna isolada é uma Series:")
print("type(df_bruto['valor']) ->", type(df_bruto["valor"]).__name__)
df_bruto.head()

df_bruto é um DataFrame com shape (8, 5)

Uma coluna isolada é uma Series:
type(df_bruto['valor']) -> Series


,id,data,cliente,categoria,valor
0,1,2026-06-01,Ana Souza,eletronicos,1200.00
1,2,01/06/2026,Bruno Lima,alimentacao,85.50
2,3,2026-06-02,Carla Dias,eletronicos,NaN
3,4,2026-06-02,Diego Reis,transporte,-30.00
4,5,2026/06/03,Ana Souza,alimentacao,42.90


In [5]:
# .info() — o método mais importante logo após ler um dado:
# quantas linhas, quais colunas, dtypes e contagem de não-nulos.
df_bruto.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         8 non-null      object
 1   data       7 non-null      object
 2   cliente    8 non-null      object
 3   categoria  8 non-null      object
 4   valor      7 non-null      object
dtypes: object(5)
memory usage: 448.0+ bytes


In [6]:
# Lendo SEM forçar str, o pandas tenta inferir tipos — mas o CSV sujo o atrapalha:
df_infer = pd.read_csv(io.StringIO(csv_bruto))
print("dtypes inferidos pelo pandas no dado BRUTO:")
print(df_infer.dtypes)
print("\nObserve: 'valor' virou float64 (por causa do campo vazio -> NaN),")
print("mas 'data' continua 'object' (texto), pois tem 3 formatos misturados.")
print("\n.describe() das colunas numéricas — um detector barato de problema:")
df_infer.describe()

dtypes inferidos pelo pandas no dado BRUTO:
id             int64
data          object
cliente       object
categoria     object
valor        float64
dtype: object

Observe: 'valor' virou float64 (por causa do campo vazio -> NaN),
mas 'data' continua 'object' (texto), pois tem 3 formatos misturados.

.describe() das colunas numéricas — um detector barato de problema:


,id,valor
count,8.00000,7.000000
mean,4.12500,243.028571
std,2.03101,431.283139
min,1.00000,-30.000000
25%,2.75000,42.900000
50%,4.50000,85.500000
75%,5.25000,179.950000
max,7.00000,1200.000000


**O que `describe()` já denuncia:** o `min` da coluna `valor` é **-30,00** (o id 4, negativo) e há apenas **7 valores não-nulos** de 8 linhas (o id 3 está vazio). Os problemas aparecem antes de qualquer transformação.

---

# 3 — Seleção e filtragem

> 📖 **No `.md`:** seção *3 — Seleção e filtragem*.

`loc`/`iloc`, seleção de colunas e — o coração do pandas — **máscaras booleanas**. Vamos usá-las para isolar exatamente as linhas problemáticas.

In [7]:
# Trabalhamos sobre o df com tipos inferidos (valor já é numérico).
df = df_infer.copy()

# Seleção de COLUNAS:
print("Uma coluna (Series):")
print(df["categoria"].head(3))
print("\nVárias colunas (DataFrame):")
print(df[["cliente", "valor"]].head(3))

Uma coluna (Series):
0    eletronicos
1    alimentacao
2    eletronicos
Name: categoria, dtype: object

Várias colunas (DataFrame):
         cliente   valor
0      Ana Souza  1200.0
1    Bruno Lima     85.5
2     Carla Dias     NaN


In [8]:
# .iloc -> por POSIÇÃO (começa em 0). As 3 primeiras linhas, todas as colunas:
print("df.iloc[0:3] — primeiras 3 linhas por posição:")
print(df.iloc[0:3])

# .loc -> por RÓTULO/CONDIÇÃO. Cliente e valor das linhas com valor > 100:
print("\ndf.loc[mascara, colunas] — clientes com valor > 100:")
print(df.loc[df["valor"] > 100, ["cliente", "valor"]])

df.iloc[0:3] — primeiras 3 linhas por posição:
   id        data        cliente    categoria   valor
0   1  2026-06-01      Ana Souza  eletronicos  1200.0
1   2  01/06/2026    Bruno Lima   alimentacao    85.5
2   3  2026-06-02     Carla Dias  eletronicos     NaN

df.loc[mascara, colunas] — clientes com valor > 100:
      cliente   valor
0   Ana Souza  1200.0
6  Elena Cruz   260.0


In [9]:
# MÁSCARA BOOLEANA: uma comparação devolve uma Series de True/False, uma por linha.
mascara_negativo = df["valor"] < 0
print("Máscara 'valor < 0' (uma booleana por linha):")
print(mascara_negativo.tolist())

print("\nLinhas com valor NEGATIVO (id 4):")
print(df[mascara_negativo])

print("\nLinhas com valor AUSENTE (id 3) — usando .isna():")
print(df[df["valor"].isna()])

# Combinando condições com & | ~  (SEMPRE entre parênteses):
print("\nValor > 0 E categoria 'alimentacao':")
print(df[(df["valor"] > 0) & (df["categoria"] == "alimentacao")])

Máscara 'valor < 0' (uma booleana por linha):
[False, False, False, True, False, False, False, False]

Linhas com valor NEGATIVO (id 4):
   id        data     cliente   categoria  valor
3   4  2026-06-02  Diego Reis  transporte  -30.0

Linhas com valor AUSENTE (id 3) — usando .isna():
   id        data     cliente    categoria  valor
2   3  2026-06-02  Carla Dias  eletronicos    NaN

Valor > 0 E categoria 'alimentacao':
   id        data        cliente    categoria  valor
1   2  01/06/2026    Bruno Lima   alimentacao   85.5
4   5  2026/06/03      Ana Souza  alimentacao   42.9
5   5  2026/06/03      Ana Souza  alimentacao   42.9


---

# 4 — Limpeza com pandas

> 📖 **No `.md`:** seção *4 — Limpeza com pandas*.

Agora tratamos, **uma a uma**, todas as sujeiras do CSV — cada problema do E1 vira um verbo de tabela. E, fiéis à lição de rastreabilidade do E1, **contamos cada descarte**.

In [10]:
# Partimos do dado totalmente bruto (tudo texto), como chega da fonte.
df = pd.read_csv(io.StringIO(csv_bruto), dtype=str)
n_inicial = len(df)
print(f"[INÍCIO] {n_inicial} linhas brutas.\n")

# (1) CONVERSÃO DE TIPOS com errors='coerce' — inválido vira NaN/NaT, não quebra.
df["valor"] = pd.to_numeric(df["valor"], errors="coerce")
df["data"] = pd.to_datetime(df["data"], errors="coerce", format="mixed", dayfirst=True)
df["id"] = df["id"].astype("int64")

print("Após conversão de tipos:")
print(f"  valores ausentes em 'valor': {df['valor'].isna().sum()}  (id 3)")
print(f"  datas inválidas/vazias em 'data': {df['data'].isna().sum()}  (id 7)")
print("\ndtypes agora corretos:")
print(df.dtypes)

[INÍCIO] 8 linhas brutas.

Após conversão de tipos:
  valores ausentes em 'valor': 1  (id 3)
  datas inválidas/vazias em 'data': 1  (id 7)

dtypes agora corretos:
id                    int64
data         datetime64[ns]
cliente              object
categoria            object
valor               float64
dtype: object


In [11]:
# (2) ESPAÇOS — str.strip() vetorizado na coluna inteira (id 2):
print("Antes:", repr(df.loc[1, "cliente"]))
df["cliente"] = df["cliente"].str.strip()
print("Depois:", repr(df.loc[1, "cliente"]))

# (3) DUPLICATAS — drop_duplicates remove a linha idêntica (id 5):
antes = len(df)
df = df.drop_duplicates()
print(f"\ndrop_duplicates: {antes} -> {len(df)} linhas (removeu 1 duplicata, id 5)")

Antes: '  Bruno Lima '
Depois: 'Bruno Lima'

drop_duplicates: 8 -> 7 linhas (removeu 1 duplicata, id 5)


In [12]:
# (4) DESCARTES com contagem — a rastreabilidade do E1, agora em pandas.
descartes = {}

# valor ausente (id 3)
mask_aus = df["valor"].isna()
descartes["valor_ausente"] = int(mask_aus.sum())

# data inválida/vazia (id 7)
mask_data = df["data"].isna()
descartes["data_invalida"] = int(mask_data.sum())

# valor negativo (id 4)
mask_neg = df["valor"] < 0
descartes["valor_negativo"] = int(mask_neg.sum())

# aplica as remoções
df_limpo = df[~(mask_aus | mask_data | mask_neg)].copy()

print(f"Descartes contados: {descartes}")
print(f"Linhas mantidas: {len(df_limpo)} de {n_inicial} brutas.")
print("\nDado limpo:")
df_limpo

Descartes contados: {'valor_ausente': 1, 'data_invalida': 1, 'valor_negativo': 1}
Linhas mantidas: 4 de 8 brutas.

Dado limpo:


,id,data,cliente,categoria,valor
0,1,2026-06-01,Ana Souza,eletronicos,1200.0
1,2,2026-06-01,Bruno Lima,alimentacao,85.5
4,5,2026-06-03,Ana Souza,alimentacao,42.9
6,6,2026-06-03,Elena Cruz,vestuario,260.0


---

# 5 — Transformação e agregação

> 📖 **No `.md`:** seção *5 — Transformação e agregação*.

Com o dado limpo, **derivamos** colunas (`assign`), **agrupamos** (`groupby().agg()`), **juntamos** com uma dimensão (`merge`) e montamos uma **pivot_table**.

In [13]:
# (a) COLUNA DERIVADA com assign — total com 18% de imposto, vetorizado:
df_limpo = df_limpo.assign(
    total_com_imposto=lambda d: (d["valor"] * 1.18).round(2)
)
print("Coluna derivada 'total_com_imposto':")
print(df_limpo[["cliente", "valor", "total_com_imposto"]])

Coluna derivada 'total_com_imposto':
      cliente   valor  total_com_imposto
0   Ana Souza  1200.0            1416.00
1  Bruno Lima    85.5             100.89
4   Ana Souza    42.9              50.62
6  Elena Cruz   260.0             306.80


In [14]:
# (b) AGREGAÇÃO com groupby().agg() — o GROUP BY do SQL.
# Responde a pergunta do fio condutor: 'qual o faturamento por categoria?'
resumo = (
    df_limpo.groupby("categoria")
    .agg(
        faturamento=("valor", "sum"),
        ticket_medio=("valor", "mean"),
        qtd=("valor", "count"),
    )
    .sort_values("faturamento", ascending=False)
)
print("Faturamento por categoria (split -> apply -> combine):")
resumo

Faturamento por categoria (split -> apply -> combine):


,faturamento,ticket_medio,qtd
categoria,,,
eletronicos,1200.0,1200.0,1
vestuario,260.0,260.0,1
alimentacao,128.4,64.2,2


In [15]:
# (c) MERGE — junção com uma pequena TABELA DE DIMENSÃO (padrão fato x dimensão).
dim_categoria = pd.DataFrame({
    "categoria": ["eletronicos", "alimentacao", "vestuario", "transporte", "servicos"],
    "tipo_gasto": ["superfluo", "essencial", "superfluo", "essencial", "essencial"],
})

antes = len(df_limpo)
df_enriquecido = df_limpo.merge(dim_categoria, on="categoria", how="left")
print(f"merge (left join): {antes} linhas antes -> {len(df_enriquecido)} depois")
print("(número de linhas preservado: o join não multiplicou nada)\n")
print(df_enriquecido[["cliente", "categoria", "tipo_gasto", "valor"]])

merge (left join): 4 linhas antes -> 4 depois
(número de linhas preservado: o join não multiplicou nada)

      cliente    categoria tipo_gasto   valor
0   Ana Souza  eletronicos  superfluo  1200.0
1  Bruno Lima  alimentacao  essencial    85.5
2   Ana Souza  alimentacao  essencial    42.9
3  Elena Cruz    vestuario  superfluo   260.0


In [16]:
# Faturamento por TIPO de gasto, possível só após o merge:
print("Faturamento por tipo de gasto (essencial x supérfluo):")
print(df_enriquecido.groupby("tipo_gasto")["valor"].sum())

# (d) PIVOT_TABLE — faturamento por categoria x data (visão de negócio):
print("\npivot_table — faturamento por categoria x data:")
pivot = df_enriquecido.pivot_table(
    index="categoria", columns="data", values="valor", aggfunc="sum", fill_value=0
)
pivot

Faturamento por tipo de gasto (essencial x supérfluo):
tipo_gasto
essencial     128.4
superfluo    1460.0
Name: valor, dtype: float64

pivot_table — faturamento por categoria x data:


data,2026-06-01,2026-06-03
categoria,,
alimentacao,85.5,42.9
eletronicos,1200.0,0.0
vestuario,0.0,260.0


---

# 6 — Pipeline do E1 refeito em pandas

> 📖 **No `.md`:** seção *6 — Pipeline do E1 refeito em pandas*.

Tudo converge aqui: o **mesmo `vendas.csv`** do E1, levado de bruto a confiável num **encadeamento declarativo** (E→T→L), salvo em **Parquet**. A prova de sucesso: o faturamento tem que **bater com o do E1**.

In [17]:
# ---------- O PIPELINE COMPLETO EM UM ENCADEAMENTO (method chaining) ----------
df_final = (
    pd.read_csv(io.StringIO(csv_bruto), dtype=str)                      # EXTRAIR
      .assign(                                                          # TRANSFORMAR
          valor=lambda d: pd.to_numeric(d["valor"], errors="coerce"),
          data=lambda d: pd.to_datetime(d["data"], errors="coerce",
                                        format="mixed", dayfirst=True),
          cliente=lambda d: d["cliente"].str.strip(),
          id=lambda d: d["id"].astype("int64"),
      )
      .drop_duplicates()                                                # remove duplicata (id 5)
      .dropna(subset=["valor", "data"])                                 # remove ausente e data vazia
      .query("valor >= 0")                                              # remove negativo (id 4)
      .assign(total_com_imposto=lambda d: (d["valor"] * 1.18).round(2)) # enriquece
      .reset_index(drop=True)
)

print(f"[E->T->L] dado confiável: {len(df_final)} linhas (4 esperadas)")
df_final

[E->T->L] dado confiável: 4 linhas (4 esperadas)


,id,data,cliente,categoria,valor,total_com_imposto
0,1,2026-06-01,Ana Souza,eletronicos,1200.0,1416.00
1,2,2026-06-01,Bruno Lima,alimentacao,85.5,100.89
2,5,2026-06-03,Ana Souza,alimentacao,42.9,50.62
3,6,2026-06-03,Elena Cruz,vestuario,260.0,306.80


In [18]:
# ---------- CARREGAR: salvar em Parquet (preserva os dtypes!) ----------
df_final.to_parquet("vendas_tratadas.parquet", index=False)

# Releitura: o Parquet devolve os tipos prontos, sem re-conversão.
df_relido = pd.read_parquet("vendas_tratadas.parquet")
print("Parquet relido — dtypes preservados (data continua data, valor continua número):")
print(df_relido.dtypes)
print("\nO consumidor recebe dado tipado, não texto para re-converter.")

Parquet relido — dtypes preservados (data continua data, valor continua número):
id                            int64
data                 datetime64[ns]
cliente                      object
categoria                    object
valor                       float64
total_com_imposto           float64
dtype: object

O consumidor recebe dado tipado, não texto para re-converter.


In [19]:
# ---------- CONFERÊNCIA: o faturamento bate com o do Encontro 1? ----------
fat = (df_final.groupby("categoria")["valor"].sum()
       .sort_values(ascending=False))

print("Faturamento por categoria (pandas, dado tratado):")
for cat, total in fat.items():
    print(f"  {cat:14s} R$ {total:>9.2f}")

# Valores de referência do Encontro 1 (Python puro):
esperado_e1 = {"eletronicos": 1200.00, "vestuario": 260.00, "alimentacao": 128.40}
bate = all(abs(fat.get(c, 0) - v) < 0.01 for c, v in esperado_e1.items())
print(f"\nBate EXATAMENTE com o Encontro 1? {bate}")
print("Mesma água, encanamento melhor: ~50 linhas de laços viraram um encadeamento.")

Faturamento por categoria (pandas, dado tratado):
  eletronicos    R$   1200.00
  vestuario      R$    260.00
  alimentacao    R$    128.40

Bate EXATAMENTE com o Encontro 1? True
Mesma água, encanamento melhor: ~50 linhas de laços viraram um encadeamento.


### Caixa de tendência de mercado — Arrow e Polars

> 📖 **No `.md`:** caixa *Tendência de mercado — Arrow e Polars*.

Duas direções do ecossistema, demonstradas a seguir: o **backend Arrow** do pandas 2.x e o **Polars** (lazy, multicore) fazendo a **mesma agregação**.

In [20]:
# (a) BACKEND ARROW no pandas 2.x: representação colunar de memória (texto/nulos eficientes).
try:
    df_arrow = pd.read_csv(io.StringIO(csv_bruto), dtype_backend="pyarrow")
    print("Lido com dtype_backend='pyarrow' — dtypes apoiados em Apache Arrow:")
    print(df_arrow.dtypes)
    print("\nÉ o MESMO pandas, com fundação de memória colunar (Arrow) por baixo.")
except Exception as e:
    print("Backend pyarrow indisponível nesta versão de pandas:", e)

Lido com dtype_backend='pyarrow' — dtypes apoiados em Apache Arrow:
id            int64[pyarrow]
data         string[pyarrow]
cliente      string[pyarrow]
categoria    string[pyarrow]
valor        double[pyarrow]
dtype: object

É o MESMO pandas, com fundação de memória colunar (Arrow) por baixo.


In [21]:
# (b) MESMA AGREGAÇÃO em pandas vs. Polars (lazy). Polars é opcional.
print("=== pandas (eager) ===")
fat_pandas = df_final.groupby("categoria")["valor"].sum().sort_values(ascending=False)
print(fat_pandas)

if TEM_POLARS:
    print("\n=== Polars (lazy: monta um plano, otimiza, então executa) ===")
    pl_df = pl.from_pandas(df_final[["categoria", "valor"]])
    fat_polars = (
        pl_df.lazy()                                  # entra no modo LAZY
        .group_by("categoria")
        .agg(pl.col("valor").sum().alias("faturamento"))
        .sort("faturamento", descending=True)
        .collect()                                    # só aqui o plano é EXECUTADO
    )
    print(fat_polars)
    print("\nMesmo resultado, filosofias diferentes: pandas executa na hora (eager);")
    print("Polars monta um plano de consulta, otimiza e só então executa (lazy).")
else:
    print("\n[Polars não instalado] O código Polars equivalente seria:")
    print('''pl_df.lazy().group_by("categoria")
     .agg(pl.col("valor").sum().alias("faturamento"))
     .sort("faturamento", descending=True).collect()''')
    print("Instale com: pip install polars")

=== pandas (eager) ===
categoria
eletronicos    1200.0
vestuario       260.0
alimentacao     128.4
Name: valor, dtype: float64

=== Polars (lazy: monta um plano, otimiza, então executa) ===
shape: (3, 2)
┌─────────────┬─────────────┐
│ categoria   ┆ faturamento │
│ ---         ┆ ---         │
│ str         ┆ f64         │
╞═════════════╪═════════════╡
│ eletronicos ┆ 1200.0      │
│ vestuario   ┆ 260.0       │
│ alimentacao ┆ 128.4       │
└─────────────┴─────────────┘

Mesmo resultado, filosofias diferentes: pandas executa na hora (eager);
Polars monta um plano de consulta, otimiza e só então executa (lazy).


---

## 🧪 Exercícios do Encontro 2

Resolva no próprio notebook, criando novas células abaixo:

1. **Seleção (seção 3).** Usando uma máscara booleana, mostre apenas as linhas do dado limpo (`df_limpo`) cujo `total_com_imposto` seja maior que 100. Quantas são?
2. **Limpeza (seção 4).** Adicione ao `csv_bruto` uma nova linha com um problema *novo* — por exemplo, um `valor` não-numérico como `"abc"`. Releia e mostre que `pd.to_numeric(..., errors="coerce")` o transforma em `NaN` sem quebrar o pipeline. Conte os ausentes antes e depois.
3. **Agregação (seção 5).** A partir de `df_enriquecido`, use `groupby().agg()` para descobrir, por `tipo_gasto`, o **faturamento total** e o **número de clientes distintos** (dica: `("cliente", "nunique")`).
4. **Pipeline (seção 6).** Reescreva o encadeamento da seção 6 trocando `.query("valor >= 0")` por `.loc[lambda d: d["valor"] >= 0]`. Confirme que o faturamento final continua batendo com o do E1.

> Dica: o melhor exercício é **quebrar uma etapa do encadeamento de propósito** (ex.: tirar o `dropna`) e observar como o erro — ou o número errado — aparece. Erros são o melhor professor de programação.

---

## Encerramento

Este notebook acompanha o documento `.md` (teoria detalhada) e os slides `.pptx` (aula expositiva) do Encontro 2.

Hoje você: mediu a **vetorização** contra o laço puro do E1, leu dados em **DataFrame/Series**, filtrou com **máscaras booleanas**, **limpou** cada sujeira do `vendas.csv` com verbos de tabela, **agregou e juntou** com `groupby`/`merge`/`pivot_table`, e **refez o pipeline do E1 em pandas** — entregando **Parquet** e chegando ao **mesmo faturamento**. Ainda espiou para onde o ecossistema caminha: **Arrow** e **Polars**.

**No Encontro 3:** SQL. Repare que `groupby`, `merge` e máscaras já têm um par exato em `GROUP BY`, `JOIN` e `WHERE` — essa ponte é o próximo passo.

**Bibliografia:** McKinney, *Python para Análise de Dados* (2022) · Reis & Housley, *Fundamentos de Engenharia de Dados* (2023) · Kleppmann, *Designing Data-Intensive Applications* (2017).